# Kaggle Competition
Due Nov 11

Team 27: Rachel Liu, Henry Lam, Yang Xiao

The PR should include
* implementation,
* experimental code,
* README,
* dependencies, and
* instructions for generating your Kaggle predictions.

Your PR should preserve meaningful development history, including experiments, debugging, and changes to your approach

## Option 1: Neural Net
0. preprocess
1. pick a **model class**, 
2. an **estimation principle**
3. turn it into a **loss**,
4. **optimize** it, 
5. then **predict and evaluate** 

In [1]:
%matplotlib inline
import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import torch
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score
from scipy.sparse import hstack

SEED = 27   # team number, so every run uses the same split and weight initialization
np.random.seed(SEED)
torch.manual_seed(SEED)

### Step 0: Preprocess Data
Make the raw data (attributes) usable via featurization, PCA.

#### (a) Load the data

In [2]:
train_df = pd.read_csv("data/train.csv")   # id, text, label (1 = AI, 0 = human)
test_df = pd.read_csv("data/test.csv")     # id, text (no labels, this is what we submit predictions for)

print("train.csv shape:", train_df.shape)
print("test.csv  shape:", test_df.shape)
print("fraction labeled AI:", train_df["label"].mean())   # 0.5 means the classes are balanced
train_df.head()

train.csv shape: (10000, 3)
test.csv  shape: (3000, 2)
fraction labeled AI: 0.5


,id,text,label
0,1,Cipriani S.A. is an Italian hotel and leisure ...,0
1,2,My Grandpa passed away 3 years ago from cancer...,0
2,3,"""Petit Beurre -- Butter Cookies\n\n1 cup (2 st...",1
3,4,"In an unprecedented move, a dedicated gamer ha...",1
4,5,Chile's copper industry has registered record ...,0


#### (b) Split the training data
1. Fit on the training set, 
2. use the held-out development set to compare models,
3. pick hyperparameters, and
4. decide when to stop training.

Since the test set has no labels the held-out dev set will be used to compare models.

In [4]:
train_set, dev_set = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df["label"]
)

y_train = train_set["label"].values
y_dev = dev_set["label"].values

print("training set size:   ", len(train_set), f"({y_train.mean()*100:.1f}% AI)")
print("development set size:", len(dev_set), f"({y_dev.mean()*100:.1f}% AI)")

training set size:    8000 (50.0% AI)
development set size: 2000 (50.0% AI)
